# AIxodia — stateless Cloudflare D1 mirror watchdog (Kaggle)

Notebook นี้ทำหน้าที่เดียว: อ่าน SCHEDULE จาก kernel output ของ Notebook หลัก แล้ว mirror `sessions/<id>` และ `config:*` ไป Cloudflare D1 ผ่าน D1 REST ตรง (`POST /client/v4/accounts/{a}/d1/database/{d}/query`). ใช้ token จาก Kaggle Secret `CF_TOKEN` เท่านั้น (REQ-046(3)/REQ-046(6) — ไม่ hardcode, ไม่อ่านจาก cell). ถ้า SCHEDULE หรือ token ไม่พร้อม จะหยุดเงียบและพิมพ์เหตุผล ไม่ยิง request เปล่า.

In [2]:
import os, time, base64, json, urllib.request, urllib.error
from datetime import datetime, timezone

CFG = json.loads(base64.b64decode('ewogICJlbmFibGVkIjogdHJ1ZSwKICAidHVubmVsIjogdHJ1ZSwKICAiY2xvdWRmbGFyZWQiOiAiY2xvdWRmbGFyZWQiLAogICJoZWFydGJlYXRFdmVyeVNlYyI6IDI1LAogICJsb29wU2xlZXBTZWMiOiAxLjAsCiAgImFkb3B0RW52IjogIkNGX1RPS0VOIiwKICAibm9kZXNVcmxUdGxTZWMiOiA2MCwKICAibWF4Qnl0ZXMiOiAxMDAwMDAsCiAgInN0YXRlUHJlZml4RmlsdGVyIjogIiIsCiAgIm1heFR1cm5zUGVyUG9zdCI6IDIwMCwKICAiZDFEYXRhYmFzZSI6ICJhaXhvZGlhIiwKICAic2VuZE9uU3RhcnQiOiB7CiAgICAidHVubmVsX3VybCI6IHRydWUsCiAgICAic2Vzc2lvbnMiOiBmYWxzZQogIH0KfQ==').decode())
CF_TOKEN = os.environ.get(CFG["adoptEnv"], "")
ENTRY = json.loads(base64.b64decode('ewogICJtb2JpbGUiOiB7CiAgICAiZW5hYmxlZCI6IHRydWUsCiAgICAiY2xvdWRmbGFyZV9hcGkiOiAiIiwKICAgICJkMV9kYXRhYmFzZSI6ICJhaXhvZGlhIiwKICAgICJsaXN0ZW4iOiAiMTI3LjAuMC4xOjE4NzkwIiwKICAgICJwdWJsaWNfbGlzdGVuIjogIiIsCiAgICAidHVubmVsIjogdHJ1ZSwKICAgICJjbG91ZGZsYXJlZCI6ICJjbG91ZGZsYXJlZCIsCiAgICAic3luY19jb25maWciOiBmYWxzZSwKICAgICJzeW5jX3Nlc3Npb25zIjogdHJ1ZQogIH0KfQ==').decode())
print("cell 1/6: config ready function=decide_source entry.mobile.listen=%s" % ENTRY["mobile"]["listen"])
print("hint: token มาจาก %s เท่านั้น (ยาว=%d)" % (CFG["adoptEnv"], len(CF_TOKEN)))

In [3]:
def api_base():
    return "https://api.cloudflare.com/client/v4"

def cf_headers(token):
    return {"Authorization": "Bearer " + token, "Content-Type": "application/json"}

def cf_get(path, token):
    req = urllib.request.Request(api_base() + path, headers=cf_headers(token))
    try:
        with urllib.request.urlopen(req, timeout=30) as r:
            return json.load(r)
    except urllib.error.HTTPError as e:
        try:
            return json.loads(e.read().decode())
        except Exception:
            return {"success": False, "errors": [{"code": e.code, "message": "http"}]}

def pick_target(token):
    accs = (cf_get("/accounts", token).get("result") or [])
    if not accs:
        return None, "accounts ว่าง — token ไม่มีสิทธิ์ Account หรือหมดอายุ"
    acc = accs[0]["id"]
    dbs = (cf_get(f"/accounts/{acc}/d1/database", token).get("result") or [])
    want = CFG.get("d1Database", "aixodia")
    db = next((d for d in dbs if d.get("name") == want), dbs[0] if dbs else None)
    if db is None:
        return None, "ไม่พบ D1 ชื่อ %r (มี: %s)" % (want, [d.get("name") for d in dbs])
    return (acc, db["uuid"] if "uuid" in db else db.get("id")), None

def d1_query(acc, dbid, token, sql, params=(), dry=False):
    body = json.dumps({"sql": sql, "params": list(params)}).encode()
    if dry:
        return {"dry": True, "bytes": len(body)}
    req = urllib.request.Request(
        f"{api_base()}/accounts/{acc}/d1/database/{dbid}/query",
        data=body, headers=cf_headers(token))
    try:
        with urllib.request.urlopen(req, timeout=30) as r:
            env = json.load(r)
    except urllib.error.HTTPError as e:
        try:
            env = json.loads(e.read().decode())
        except Exception:
            env = {"success": False}
    ok = bool(env.get("success")) and all((s.get("success", True) for s in (env.get("result") or [])))
    if not ok:
        raise RuntimeError("D1 query ล้มเหลว: %r" % (env.get("errors"),))
    return env

def emit(event, **fields):
    rec = {"t": datetime.now(timezone.utc).isoformat(), "event": event}
    rec.update(fields)
    print(json.dumps(rec, ensure_ascii=False))
    return rec

print("cell 2/6: network helpers ready function=cf_get,pick_target,d1_query")

In [4]:
def schedule_text():
    txt = globals().get("SCHEDULE", None)
    if isinstance(txt, str) and txt.strip():
        return txt, "var"
    texts = []
    try:
        ip = get_ipython()
        outs = getattr(ip, "output_hist", {}) or {}
        it = outs.values() if hasattr(outs, "values") else outs
        for cell_out in it:
            try:
                for o in getattr(cell_out, "outputs", []):
                    t = (o.get("text") or "") + (o.get("data", {}).get("text/plain", "") or "")
                    if isinstance(t, str) and t.strip():
                        texts.append(t)
            except Exception:
                continue
    except Exception:
        pass
    blob = chr(10).join(texts)
    return blob, "outputs" if blob.strip() else "missing"

def validate_token(token):
    if not token:
        return "CF_TOKEN ว่าง — ไปที่ Add-ons → Secrets แล้วเพิ่มชื่อนี้ก่อน"
    if len(token.strip()) < 20:
        return "CF_TOKEN สั้นผิดปกติ (%d ตัว) — ตรวจว่า copy มาครบ" % len(token.strip())
    return None

print("cell 3/6: discover helpers ready function=schedule_text,validate_token")

In [5]:
def parse_schedule(text):
    sessions, configs = {}, {}
    current = None
    for raw in (text or "").splitlines():
        line = raw.strip()
        if line.startswith("# SESSION:"):
            current = line.split(":", 1)[1].strip()
            sessions.setdefault(current, [])
            continue
        if line.startswith("# CONFIG:"):
            key = line.split(":", 1)[1].strip()
            configs[key] = configs.get(key, "")
            current = ("config", key)
            continue
        if line.startswith("# END"):
            current = None
            continue
        if not line or current is None:
            continue
        if isinstance(current, tuple) and current[0] == "config":
            configs[current[1]] += raw + chr(10)
        else:
            sessions[current].append(raw)
    out_s = {k: chr(10).join(v).strip() for k, v in sessions.items()}
    out_c = {k: v.strip() for k, v in configs.items()}
    return out_s, out_c

def make_ops(sessions, configs, max_turns=200):
    ops = []
    for sid, body in sessions.items():
        turns = [t for t in [x.strip() for x in body.split(chr(10))] if t][:max_turns]
        ops.append(("sessions_put", sid, turns))
    for key, val in configs.items():
        ops.append(("config_put", "config:" + key, val))
    return ops

print("cell 4/6: schedule parser ready function=parse_schedule,make_ops")

In [6]:
def plan(sessions, configs, nodes_url=None, have_token=True):
    ops = make_ops(sessions, configs, CFG.get("maxTurnsPerPost", 200))
    if not ops:
        return {"action": "skip", "reason": "SCHEDULE ไม่มี SESSION/CONFIG ที่ parse ได้"}
    if not have_token:
        return {"action": "wait", "reason": "รอ CF_TOKEN จาก Secrets", "ops": len(ops)}
    action = "push"
    if nodes_url:
        return {"action": action, "reason": "พร้อมยิง", "ops": len(ops),
                "tunnel_url_seen": nodes_url}
    return {"action": action, "reason": "พร้อมยิง (ยังไม่เห็น nodes URL จะประกาศหลัง push สำเร็จ)",
            "ops": len(ops), "tunnel_url_seen": None}

def push(ops, target, token, limit_bytes=100000):
    acc, dbid = target
    results = []
    for kind, key, val in ops:
        if kind == "config_put":
            if len(val.encode()) > limit_bytes:
                results.append({"key": key, "ok": False, "skipped": "oversize"})
                continue
            d1_query(acc, dbid, token,
                "INSERT INTO state(key,value,updated_at) VALUES(?,?,unixepoch()) "
                "ON CONFLICT(key) DO UPDATE SET value=excluded.value,updated_at=unixepoch()",
                (key, val))
            results.append({"key": key, "ok": True})
        else:
            body = json.dumps({"turns": val}, ensure_ascii=False)
            if len(body.encode()) > limit_bytes:
                results.append({"key": "sessions/" + key, "ok": False, "skipped": "oversize"})
                continue
            d1_query(acc, dbid, token,
                "INSERT INTO state(key,value,updated_at) VALUES(?,?,unixepoch()) "
                "ON CONFLICT(key) DO UPDATE SET value=excluded.value,updated_at=unixepoch()",
                ("sessions/" + key, body))
            results.append({"key": "sessions/" + key, "ok": True})
    return results

def announce(acc, dbid, token, url, version="kaggle-aixodia"):
    d1_query(acc, dbid, token,
        "INSERT INTO nodes(id,tunnel_url,version,heartbeat) VALUES(?,?,?,unixepoch()) "
        "ON CONFLICT(id) DO UPDATE SET tunnel_url=excluded.tunnel_url,version=excluded.version,heartbeat=unixepoch()",
        ("ai", url, version))

def read_nodes_url(acc, dbid, token):
    env = d1_query(acc, dbid, token, "SELECT tunnel_url,heartbeat FROM nodes WHERE id='ai' LIMIT 1")
    try:
        rows = (((env.get("result") or [])[0] or {}).get("results") or [])
        if rows and rows[0].get("tunnel_url"):
            return rows[0]["tunnel_url"], rows[0].get("heartbeat")
    except Exception:
        pass
    return None, None

print("cell 5/6: planner ready function=plan,push,announce,read_nodes_url")

In [7]:
err = validate_token(CF_TOKEN)
text, src = schedule_text()
print("source=%s chars=%d token_len=%d" % (src, len(text), len(CF_TOKEN)))
sessions, configs = parse_schedule(text)

target, target_err = (None, err) if err else pick_target(CF_TOKEN)
nodes_url, nodes_hb = (None, None)
if target and not err:
    try:
        nodes_url, nodes_hb = read_nodes_url(*target, CF_TOKEN)
    except Exception as e:
        emit("d1_nodes_read_failed", error=str(e)[:200])

decision = plan(sessions, configs, nodes_url, have_token=(err is None and target is not None))
print("decision:", json.dumps(decision, ensure_ascii=False))

if decision["action"] == "push":
    pushed = push(make_ops(sessions, configs, CFG.get("maxTurnsPerPost", 200)),
                  target, CF_TOKEN, CFG.get("maxBytes", 100000))
    emit("push_done", ok=sum(1 for r in pushed if r.get("ok")),
         skipped=[r["key"] for r in pushed if not r.get("ok")],
         source=src, nodes_url_seen=nodes_url)
    if nodes_url:
        try:
            announce(*target, CF_TOKEN, nodes_url)
            emit("announce_done", tunnel_url=nodes_url)
        except Exception as e:
            emit("announce_failed", error=str(e)[:200])
elif decision["action"] == "wait":
    emit("waiting", reason=decision["reason"], ops=decision.get("ops", 0))
else:
    emit("skipped", reason=decision["reason"])

## Phase A — tests (CPU)

clone `Tulipskun/ai` (public) ลง `/tmp/aixodia`, ติดตั้ง Go ตาม `go.mod`, แล้วรัน `go test ./...`
ถ้า test แดง kernel จะหยุดตรงนี้ ไม่ไป Phase B

In [ ]:
import json, os, re, shutil, subprocess, tarfile, urllib.request
from datetime import datetime, timezone

def emit(event, **fields):
    rec = {"t": datetime.now(timezone.utc).isoformat(), "event": event}
    rec.update(fields)
    print(json.dumps(rec, ensure_ascii=False), flush=True)

WORK = "/tmp/aixodia"
REPO = os.path.join(WORK, "ai")
os.makedirs(WORK, exist_ok=True)

def sh(cmd, **kw):
    return subprocess.run(cmd, capture_output=True, text=True, **kw)

# 1. clone public repo (no auth needed)
if not os.path.isdir(os.path.join(REPO, ".git")):
    emit("clone_start", repo="Tulipskun/ai-engine")
    r = sh(["git", "clone", "--depth", "1", "https://github.com/Tulipskun/ai-engine", REPO])
    print((r.stdout + r.stderr)[-2000:], flush=True)
    if r.returncode != 0:
        emit("clone_failed", exit=r.returncode)
        raise SystemExit(3)
else:
    sh(["git", "-C", REPO, "fetch", "--depth", "1", "origin", "main"])
    sh(["git", "-C", REPO, "reset", "--hard", "origin/main"])
sha = sh(["git", "-C", REPO, "rev-parse", "--short", "HEAD"]).stdout.strip()
emit("repo_ready", sha=sha)

# 2. go toolchain (version from go.mod)
env = dict(os.environ)
env["GOCACHE"] = os.path.join(WORK, "gocache")
env["GOPATH"] = os.path.join(WORK, "gopath")
env["GOFLAGS"] = "-count=1"
go_bin = shutil.which("go")
r = sh(["go", "version"]) if go_bin else subprocess.CompletedProcess(["go"], 1, "", "go not in PATH")
if r.returncode != 0:
    m = re.search(r"^go (\S+)", open(os.path.join(REPO, "go.mod")).read(), re.M)
    gover = m.group(1) if m else "1.25.0"
    emit("go_install_start", version=gover)
    tgz = os.path.join(WORK, "go.tgz")
    urllib.request.urlretrieve(f"https://go.dev/dl/go{gover}.linux-amd64.tar.gz", tgz)
    with tarfile.open(tgz) as tf:
        tf.extractall(os.path.join(WORK, "gotools"))
    os.remove(tgz)
    env["PATH"] = os.path.join(WORK, "gotools", "go", "bin") + ":" + env["PATH"]
    r = sh(["go", "version"], env=env)
print("go:", (r.stdout or r.stderr).strip(), flush=True)
if r.returncode != 0:
    emit("go_missing")
    raise SystemExit(3)
globals()["GO_ENV"] = env

# 3. tests (fail here -> kernel stops, no service)
emit("test_start", timeout="25m")
p = subprocess.Popen(["go", "test", "-timeout", "25m", "./..."],
                     cwd=REPO, env=env, stdout=subprocess.PIPE,
                     stderr=subprocess.STDOUT, text=True, bufsize=1)
tail = []
for line in p.stdout:
    s = line.rstrip()
    tail.append(s)
    if len(tail) > 200:
        tail.pop(0)
    if s.startswith(("ok ", "FAIL", "---", "panic")):
        print(s, flush=True)
rc = p.wait()
emit("test_done", ok=(rc == 0), exit=rc, tail=tail[-40:])
print(f"TEST_EXIT={rc}", flush=True)
if rc != 0:
    raise SystemExit(3)


## Phase B — daemon service (CPU, สูงสุด ~11.5 ชม.)

build `ai-engine` จากโค้ดที่ผ่าน Phase A แล้วรันพร้อม quick tunnel
daemon บูตแบบไม่มี secret: token/keys มาจากมือถือครั้งแรกที่เชื่อม (D1 hydrate)
cell นี้ดัก `TUNNEL_URL` จาก log แล้วพิมพ์ซ้ำให้หาง่าย — เอา URL นี้ไปตั้งในแอป

In [ ]:
import json, os, re, shutil, stat, subprocess, threading, time, urllib.request
from datetime import datetime, timezone

def emit(event, **fields):
    rec = {"t": datetime.now(timezone.utc).isoformat(), "event": event}
    rec.update(fields)
    print(json.dumps(rec, ensure_ascii=False), flush=True)

WORK = "/tmp/aixodia"
REPO = os.path.join(WORK, "ai")
env = dict(globals().get("GO_ENV", os.environ))
TIME_CAP = 41400  # 11.5h < 12h kernel limit

# 1. state + entry.json (ENTRY from cell 1, carries no secret)
STATE = os.path.join(os.path.expanduser("~"), ".local", "share", "ai")
os.makedirs(os.path.join(STATE, "config"), exist_ok=True)
mobile = dict(ENTRY["mobile"])
# Fresh boot has no local config: hydrate provider/system catalogue from D1
# on the first verified handshake (REQ-046(4)). PushConfig skips missing
# files, so an empty state can never clobber D1 on shutdown.
mobile["sync_config"] = True
with open(os.path.join(STATE, "config", "entry.json"), "w") as f:
    json.dump({"mobile": mobile}, f, indent=2)
emit("entry_written", listen=ENTRY["mobile"]["listen"], tunnel=ENTRY["mobile"]["tunnel"])

# 2. build daemon
DAEMON = os.path.join(WORK, "ai-engine")
# -ldflags stamps the build sha into the binary, and the daemon writes it
# into the D1 nodes row, so a D1 query names the build that is serving.
SHA = subprocess.run(["git", "-C", REPO, "rev-parse", "--short", "HEAD"],
                    capture_output=True, text=True).stdout.strip() or "unknown"
r = subprocess.run(["go", "build", "-ldflags", "-X main.version=" + SHA,
                    "-o", DAEMON, "./cmd/ai-engine"], cwd=REPO,
                   env=env, capture_output=True, text=True)
print((r.stdout + r.stderr)[-3000:], flush=True)
if r.returncode != 0 or not os.path.isfile(DAEMON):
    emit("build_failed", exit=r.returncode)
    raise SystemExit(4)
emit("build_ok", path=DAEMON)

# 3. cloudflared binary (daemon spawns `cloudflared` from PATH)
BINDIR = os.path.join(WORK, "bin")
os.makedirs(BINDIR, exist_ok=True)
cf = shutil.which("cloudflared") or os.path.join(BINDIR, "cloudflared")
if not (shutil.which("cloudflared") or os.path.isfile(cf)):
    emit("cloudflared_download_start")
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", cf)
    os.chmod(cf, os.stat(cf).st_mode | stat.S_IEXEC)
env["PATH"] = BINDIR + ":" + env["PATH"]
print("cloudflared:", subprocess.run(
    ["cloudflared", "--version"], capture_output=True, text=True, env=env
).stdout.strip()[:120], flush=True)

# 4. run, stream logs, re-print tunnel URL, heartbeat until time cap
emit("service_start", cap_s=TIME_CAP,
     note="set the app endpoint to TUNNEL_URL below; phone owns keys")
proc = subprocess.Popen([DAEMON], stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1, env=env)
url_re = re.compile(r"https://[A-Za-z0-9.-]+\.trycloudflare\.com")
seen, last, t0, last_hb = [], [], time.time(), 0
def reader():
    for line in proc.stdout:
        s = line.rstrip()
        last.append(s)
        if len(last) > 200:
            last.pop(0)
        print(s, flush=True)
        m = url_re.search(s)
        if m and m.group(0) not in seen:
            seen.append(m.group(0))
            print(f"\n===== TUNNEL_URL={m.group(0)} =====\n", flush=True)
            emit("tunnel_url", url=m.group(0))
threading.Thread(target=reader, daemon=True).start()
while True:
    rc = proc.poll()
    if rc is not None:
        emit("service_down", exit=rc, tail=last[-30:])
        raise SystemExit(4)
    up = time.time() - t0
    if up >= TIME_CAP:
        emit("service_done", reason="time_cap_11_5h", uptime_s=int(up))
        proc.terminate()
        try:
            proc.wait(timeout=20)
        except subprocess.TimeoutExpired:
            proc.kill()
        break
    if up - last_hb >= 300:
        last_hb = up
        emit("heartbeat", uptime_s=int(up), tunnel=seen[-1] if seen else None)
    time.sleep(5)
